# Bubble segmentation: preprocessing + Cellpose-SAM + circle fitting

This notebook segments the bubbles in a single frame and measures the **radius / area of every bubble**,
including bubbles that are partly hidden behind other bubbles or behind static structures (fibres, film wrinkles).

**Pipeline** (all functions live in `bubble_seg.py`, all tunable values in the **CONFIG** cell):

| step | what | why |
|---|---|---|
| 1 | mask burned-in overlays (timestamp, scale bar), read pixel size from the scale bar | text and the bar would otherwise be "segmented" |
| 2 | masked background flattening → denoise → contrast stretch → CLAHE; **rim map** = dark-ridge (Sato) filter, expressed as a noise z-score | removes the uneven illumination; the rim map makes thresholds independent of brightness |
| 3 | static-structure mask (long straight ridges in one frame, or ridges present in most frames of the series) | stops fibres from producing false bubbles / rim points |
| 4 | **Cellpose-SAM** (zero-shot), optionally several passes at different diameters | robust instance segmentation of the very wide size range |
| 5 | robust **circle fit** (RANSAC) of each mask, refined on the rim map; optional ridge-Hough "completion" for large bubbles Cellpose missed | Cellpose masks cannot overlap, so a hidden bubble only gets its visible part as a mask; the circle fit recovers the full bubble |
| 6 | quality scores (arc support, ring contrast, visible fraction) + duplicate suppression | one clean table per frame, ready for tracking over time |

**Do you need to train?** Start zero-shot (this notebook). Cellpose-SAM is a generalist that handles outline-type objects
reasonably well, and the circle-fit/QC stage tolerates fragmented or partial masks. Look at the overlays in step 6:
if recall on overlapping/nested bubbles or on the fibre region is not good enough, use step 8. Correct the masks of
~10–20 tiles (a few hundred bubbles) in the Cellpose GUI and fine-tune for 100–300 epochs. That takes minutes on a
Maxwell GPU, and it usually helps a lot for an unusual image type like this one.

**Note on your data:** the JPGs are ~5× upsampled (blocky ~5 px pixels, consistent with a 400×250 high-speed sensor) and
JPEG-compressed with text burned in. If you have the raw frames (e.g. the 400×250 TIFF/HDF5 export), use those and set
`work_scale=1.0` and the length parameters ÷5. That gives better accuracy and is much faster.

## 0. Environment (DESY Maxwell)

One-time setup on a Maxwell login node (or in a JupyterHub terminal):
```bash
module load maxwell python/3.11            # or any recent python module (check: module avail python)
python -m venv ~/venvs/foam && source ~/venvs/foam/bin/activate
pip install -r requirements.txt           # in this repo; pulls cellpose>=4 (Cellpose-SAM) + torch
python -m ipykernel install --user --name foam --display-name "foam (cellpose)"
# download the Cellpose-SAM weights once (~1.2 GB) where internet is available (e.g. a login node).
# gpu=False here only because login nodes have no GPU; it is just a download.
# Segmentation in this notebook runs on the GPU via cp_gpu=True in CONFIG.
python -c "from cellpose import models; models.CellposeModel(gpu=False)"
```
Then start a JupyterHub session on a **GPU** partition (max-jhub.desy.de → a `maxgpu`/`allgpu` job profile) and
pick the `foam (cellpose)` kernel. CPU also works but is slow (~1 min per pass per frame).

In [ ]:
import os, glob, time, dataclasses
# where Cellpose stores/looks for weights: set BEFORE importing cellpose (e.g. a beegfs/dust path on Maxwell)
os.environ.setdefault("CELLPOSE_LOCAL_MODELS_PATH", os.path.expanduser("~/.cellpose/models"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%load_ext autoreload
%autoreload 2
import bubble_seg as bs

try:
    import torch
    print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available(),
          "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")
except ImportError:
    print("torch not installed - Cellpose will not run (watershed/Hough still work)")
plt.rcParams["figure.dpi"] = 90

## CONFIG: everything you may want to tweak

All lengths are in **original image pixels** (they are rescaled internally by `work_scale`).
Re-run from this cell downwards after changing anything. Defaults are tuned on frame `tid0_000`.

In [ ]:
cfg = bs.Config(
    # ---------------- I/O ----------------
    image_path="animations/jpg_r563_svd_normalised_tr050/r563_svd_normalised_tr050_tid0_000.jpg",
    output_dir="outputs",
    frame_glob="animations/jpg_r563_svd_normalised_tr050/*.jpg",

    # ---------------- pixel size ----------------
    um_per_px=None,                 # None -> measured from the burned-in scale bar
    scale_bar_um=100.0,
    scale_bar_box=(0.85, 1.0, 0.80, 1.0),   # (y0, y1, x0, x1) fractions searched for the bar

    # ---------------- regions to ignore ----------------
    exclude_boxes=[(0.00, 0.075, 0.80, 1.00),    # timestamp (fractions y0,y1,x0,x1)
                   (0.89, 1.00, 0.86, 1.00)],    # scale bar
    exclude_polygons=[],            # e.g. [[(x0,y0), (x1,y1), (x2,y2), ...]] in original px

    # ---------------- static structures ----------------
    static_mode="lines",            # 'none' | 'lines' (single frame) | 'temporal' (whole series) | 'both'
    line_min_length=120.0, line_max_gap=20.0, line_ridge_z=3.0, line_width=16.0,
    temporal_n_frames=40, temporal_percentile=20.0, temporal_ridge_z=3.0,
    static_dilate=4.0,

    # ---------------- preprocessing ----------------
    work_scale=0.5,                 # 0.5 is lossless here (images are ~5x upsampled) and 4x faster
    bg_sigma=120.0,                 # illumination background scale (>> largest rim thickness)
    bg_mode="divide",               # 'divide' | 'subtract'
    denoise="gaussian",             # 'gaussian' | 'nlm' | 'tv' | 'none'
    denoise_sigma=5.0, nlm_h=0.8, tv_weight=0.05,
    stretch_percentiles=(0.5, 99.5),
    use_clahe=True, clahe_kernel=256.0, clahe_clip=0.01,
    ridge_sigmas=(4.0, 6.0, 8.0),   # ~half the rim thickness

    # ---------------- Cellpose-SAM ----------------
    use_cellpose=True,
    cp_model="cpsam",               # or path to your fine-tuned model (step 8)
    cp_gpu=True,
    cp_input="clahe",               # 'norm' | 'clahe' | 'ridge' | 'clahe+ridge'
    cp_diameters=[None, 120.0],     # one pass per entry (original px); None = no rescaling
    cp_flow_threshold=0.4,          # raise (0.6-1.0) to keep more, less-perfect masks
    cp_cellprob_threshold=0.0,      # lower (-1..-4) for more / larger masks, raise for fewer
    cp_min_size=15,                 # working px
    cp_max_size_fraction=0.4,
    cp_tile_norm_blocksize=0,       # e.g. 200 to let Cellpose normalise locally as well
    cp_augment=False, cp_batch_size=8, cp_niter=None,

    # ---------------- classical baseline (optional) ----------------
    use_watershed=False,            # True: add watershed-of-rim-map candidates (no deep learning)
    ws_rim_z=2.0, ws_min_distance=8.0, ws_min_inner_radius=6.0, ws_min_area=120.0,

    # ---------------- circle fit + QC ----------------
    rim_z=2.5,                      # rim map z-score that counts as "on a rim"
    rim_search=2.0,                 # radial tolerance (px) when scoring a circle against the rim map
    ransac_tol=4.0, ransac_iters=300,
    min_radius=5.0, max_radius=400.0,
    min_arc_support_mask=0.35, min_ring_contrast_mask=1.0,     # Cellpose / watershed candidates
    min_arc_support_hough=0.70, min_ring_contrast_hough=2.0,   # Hough candidates (stricter)
    min_mask_circle_iou=0.05, max_r_over_mask_radius=4.0,     # mask must overlap its circle
    min_visible_frac=0.4,
    dup_frac=0.3,

    # ---------------- Hough completion ----------------
    use_hough=True,
    hough_radii=(25.0, 220.0), hough_radius_step=2.0,
    hough_edge_z=3.0, hough_peak_threshold=0.12, hough_peaks_per_radius=200,
)
os.makedirs(cfg.output_dir, exist_ok=True)

## 1. Load the frame, mask overlays, measure the pixel size

In [ ]:
img = bs.load_image(cfg.image_path)
valid = bs.build_valid_mask(img.shape, cfg)
um_per_px = cfg.um_per_px or bs.measure_scale_bar(img, cfg)
print(f"image {img.shape}, pixel size: {um_per_px} um/px" + ("  (from scale bar)" if cfg.um_per_px is None else ""))

fig, ax = plt.subplots(1, 2, figsize=(16, 5))
ax[0].imshow(img, cmap="gray"); ax[0].set_title("raw")
ax[1].imshow(np.where(valid, img, np.nan), cmap="gray"); ax[1].set_title("excluded regions blanked")
for a in ax: a.axis("off")
plt.tight_layout()

## 2. Preprocessing

* `flattened + denoised` should look evenly lit. If large-scale shading remains, lower `bg_sigma`; if big bubbles
  get a halo, raise it.
* In the `rim map`, bubble outlines should be bright and the empty background dark. Thicker rims → larger
  `ridge_sigmas`. Noisy background → larger `denoise_sigma` or `denoise="nlm"`.

In [ ]:
t = time.time()
pre = bs.preprocess(img, valid, cfg)
print(f"preprocessing: {time.time() - t:.1f}s, working size {pre['work'].shape}")

## 3. Static structures (fibres, wrinkles)

In [ ]:
temporal = None
if cfg.static_mode in ("temporal", "both"):
    frames = sorted(glob.glob(cfg.frame_glob))
    print(f"computing temporal static mask from {min(len(frames), cfg.temporal_n_frames)} of {len(frames)} frames ...")
    temporal = bs.temporal_static_mask(frames, cfg, valid)
static = bs.build_static_mask(pre, cfg, valid, temporal)
print(f"static mask covers {100 * static.mean():.1f}% of the image")
bs.plot_preprocessing(pre, static);

## 4. Cellpose-SAM

### 4a. Load the model

In [ ]:
model = None
if cfg.use_cellpose:
    t = time.time()
    model = bs.load_cellpose_model(cfg)
    print(f"loaded {cfg.cp_model} on {model.device} in {time.time() - t:.1f}s")
    if cfg.cp_gpu and model.device.type != "cuda":
        print("WARNING: cp_gpu=True but Cellpose is running on CPU - check torch.cuda.is_available() (see README)")
cp_in = bs.cellpose_input(pre, cfg)

### 4b. (optional) Quick parameter sweep on a crop

Set `RUN_SWEEP = True` to compare settings side by side on a small region before running the full frame.
`TUNE_ROI` is `(x0, x1, y0, y1)` in **original** pixels.

In [ ]:
RUN_SWEEP = False
TUNE_ROI = (900, 1700, 450, 1250)
SWEEP = [  # each entry overrides cfg fields
    dict(cp_input="clahe"),
    dict(cp_input="norm"),
    dict(cp_input="clahe+ridge"),
    dict(cp_input="clahe", cp_flow_threshold=0.8, cp_cellprob_threshold=-2.0),
]

if RUN_SWEEP and model is not None:
    s = pre["scale"]
    x0, x1, y0, y1 = (int(round(v * s)) for v in TUNE_ROI)
    fig, axs = plt.subplots(len(cfg.cp_diameters), len(SWEEP), figsize=(5 * len(SWEEP), 5 * len(cfg.cp_diameters)), squeeze=False)
    for j, over in enumerate(SWEEP):
        c = dataclasses.replace(cfg, **over)
        x = bs.cellpose_input(pre, c)[y0:y1, x0:x1]
        for i, p in enumerate(bs.run_cellpose(model, x, c, s)):
            ax = axs[i, j]
            ax.imshow(pre["clahe"][y0:y1, x0:x1], cmap="gray")
            b = bs.segmentation.find_boundaries(p["labels"], mode="inner")
            ax.imshow(np.ma.masked_where(~b, b), cmap="autumn", alpha=0.9)
            ax.set_title(f"{over}\ndiam={p['diameter']}: {p['labels'].max()} masks", fontsize=8)
            ax.axis("off")
    plt.tight_layout()

### 4c. Run Cellpose on the full frame

In [ ]:
passes = []
if cfg.use_cellpose:
    t = time.time()
    passes = bs.run_cellpose(model, cp_in, cfg, pre["scale"])
    print(f"Cellpose: {time.time() - t:.1f}s;", ", ".join(f"diam={p['diameter']}: {p['labels'].max()} masks" for p in passes))
if cfg.use_watershed:
    passes.append(dict(diameter="watershed", labels=bs.watershed_labels(pre, static, cfg), flows=None))
if passes:
    bs.plot_masks(pre, passes, figsize=(9 * len(passes), 6));

## 5. Circle fitting, Hough completion, QC

For every mask: boundary pixels that lie on a rim (and are not static) → RANSAC circle → re-fit to the
rim pixels in an annulus around it. Each candidate is scored on

* `arc_support`: fraction of the *visible* circumference lying on a rim (1 = complete outline),
* `ring_contrast`: rim strength on the circle minus just inside/outside it (rejects circles drawn through clutter),
* `visible_frac`: fraction of the circumference not excluded / static / outside the image,
* `mask_circle_iou`: how disc-like the Cellpose mask itself is (low = partial/occluded mask; informative only).

In [ ]:
rng = np.random.default_rng(cfg.random_seed)
cands = []
t = time.time()
for p in passes:
    src = "watershed" if p["diameter"] == "watershed" else "cellpose"
    cands += bs.masks_to_candidates(p["labels"], pre, static, cfg, source=src,
                                    pass_diameter=None if src == "watershed" else p["diameter"], rng=rng)
print(f"mask candidates: {len(cands)} ({time.time() - t:.1f}s)")
if cfg.use_hough:
    t = time.time()
    h = bs.hough_candidates(pre, static, cfg)
    cands += h
    print(f"Hough candidates passing scores: {len(h)} ({time.time() - t:.1f}s)")

allc = bs.add_physical_units(bs.select_bubbles(cands, cfg), um_per_px)
bubbles = allc[allc["selected"]].reset_index(drop=True)
bubbles.insert(0, "bubble_id", np.arange(len(bubbles)))
print(f"\n{len(bubbles)} bubbles selected")
print(allc.groupby("source")[["qc_pass", "selected"]].sum())

## 6. Inspect

Green = Cellpose, blue = watershed, orange = Hough completion, red dashed = candidates rejected by QC.
* Real bubbles missing and **red** circles on them → QC too strict (`min_arc_support_*`, `min_ring_contrast_*`).
* No candidate at all on a visible bubble → segmentation problem (Cellpose settings, `cp_diameters`, or fine-tune).
* False circles → raise the QC thresholds, or exclude the region / improve the static mask.

In [ ]:
bs.plot_bubbles(img, bubbles, rejected=allc[~allc["qc_pass"]]);

In [ ]:
ZOOM_ROI = (900, 1700, 450, 1250)    # (x0, x1, y0, y1) original px
bs.plot_bubbles(img, bubbles, roi=ZOOM_ROI, figsize=(10, 10), lw=1.2);

In [ ]:
cols = ["bubble_id", "source", "x", "y", "r"] + (["r_um", "area_circle_um2"] if um_per_px else ["area_circle_px2"]) \
       + ["arc_support", "ring_contrast", "visible_frac", "mask_circle_iou"]
display(bubbles[[c for c in cols if c in bubbles]].round(3).head(20))

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
rcol, unit = ("r_um", "µm") if um_per_px else ("r", "px")
for src, g in bubbles.groupby("source"):
    ax[0].hist(g[rcol], bins=40, alpha=0.6, label=src)
ax[0].set_xlabel(f"radius [{unit}]"); ax[0].set_ylabel("count"); ax[0].legend()
ax[1].scatter(bubbles[rcol], bubbles["arc_support"], c=bubbles["ring_contrast"], s=12, cmap="viridis")
ax[1].set_xlabel(f"radius [{unit}]"); ax[1].set_ylabel("arc support"); ax[1].set_title("colour: ring contrast")
plt.tight_layout()

## 7. Save results

In [ ]:
stem = os.path.splitext(os.path.basename(cfg.image_path))[0]
bubbles.to_csv(os.path.join(cfg.output_dir, f"{stem}_bubbles.csv"), index=False)
allc.to_csv(os.path.join(cfg.output_dir, f"{stem}_all_candidates.csv"), index=False)
for p in passes:
    np.save(os.path.join(cfg.output_dir, f"{stem}_masks_{p['diameter']}.npy"), p["labels"])
cfg.to_json(os.path.join(cfg.output_dir, f"{stem}_config.json"))
bs.plot_bubbles(img, bubbles).savefig(os.path.join(cfg.output_dir, f"{stem}_overlay.png"), dpi=150)
plt.close("all")
print("saved to", os.path.abspath(cfg.output_dir))

## 8. (optional) Fine-tune Cellpose-SAM on your bubbles

1. Export tiles (preprocessed Cellpose input + current masks as a starting point).
2. Correct them in the Cellpose GUI (`pip install "cellpose[gui]"`, run `cellpose`, open a tile, *File → load masks*
   the `_masks.tif`, fix, *save masks as PNG/TIF* keeping the `_masks` suffix), or in napari.
   Cellpose masks can't overlap, so for a hidden bubble label only its **visible** part. The circle fit completes it.
3. Train (GPU strongly recommended), then set `cp_model` in CONFIG to the saved model path and re-run.

In [ ]:
EXPORT_TILES = False
ANNOT_DIR = os.path.join(cfg.output_dir, "annotation_tiles")
if EXPORT_TILES:
    # start from the pass with the most masks (you will correct them anyway)
    lab0 = max(passes, key=lambda p: p["labels"].max())["labels"] if passes else np.zeros(pre["work"].shape, np.int32)
    x_exp = cp_in if cp_in.ndim == 2 else cp_in[..., 0]
    files = bs.export_tiles_for_annotation(x_exp, lab0, ANNOT_DIR, tile=256, n_tiles=12, valid=pre["valid"])
    print(f"exported {len(files)} tiles to {ANNOT_DIR}")

In [ ]:
RUN_TRAINING = False
if RUN_TRAINING:
    from cellpose import io as cpio, train
    TRAIN_DIR = cfg.output_dir
    os.makedirs(os.path.join(TRAIN_DIR, "models"), exist_ok=True)   # cellpose saves to <save_path>/models/
    imgs, lbls, names, _, _, _ = cpio.load_train_test_data(ANNOT_DIR, mask_filter="_masks")
    model_path, train_losses, test_losses = train.train_seg(
        model.net, train_data=imgs, train_labels=lbls,
        n_epochs=200, learning_rate=1e-5, weight_decay=0.1, batch_size=1,
        save_path=TRAIN_DIR, model_name="cpsam_bubbles")
    print("saved fine-tuned model:", model_path, "-> set cfg.cp_model to this path")

## 9. Next: the whole series

`bs.process_frame` runs steps 1–5 for one file. The loop below processes every frame with the current CONFIG and
writes one table (one row per bubble per frame), ready for linking bubbles over time (e.g. `trackpy.link` on
`x, y` with `r` as an extra feature). For ~1000 frames, run it as a SLURM batch job on a GPU node
(it also works as a plain Python script).

In [ ]:
RUN_BATCH = False
if RUN_BATCH:
    frames = sorted(glob.glob(cfg.frame_glob))
    temporal_b = bs.temporal_static_mask(frames, cfg, valid) if cfg.static_mode in ("temporal", "both") else None
    tables = []
    for k, f in enumerate(frames):
        b, _, _ = bs.process_frame(f, cfg, model=model, temporal_static=temporal_b, um_per_px=um_per_px, keep_debug=False)
        b.insert(0, "frame_idx", k)
        tables.append(b)
        if k % 50 == 0:
            print(f"{k}/{len(frames)}  {len(b)} bubbles")
    series = pd.concat(tables, ignore_index=True)
    series.to_csv(os.path.join(cfg.output_dir, "all_frames_bubbles.csv"), index=False)